### **Step 1 — The Corpus**

In [1]:
# Our tiny training corpus
corpus = [
    "low",
    "low",
    "low",
    "lower",
    "lowest",
    "newer",
    "wider"
]

print(corpus)
# ['low', 'low', 'low', 'lower', 'lowest', 'newer', 'wider']

['low', 'low', 'low', 'lower', 'lowest', 'newer', 'wider']


### **Step 2 — Split Words into Characters + End Marker**

In [2]:
def get_vocab(corpus):
    vocab = {}
    for word in corpus:
        # Split each word into chars, add </w> at end
        token = " ".join(list(word)) + " </w>"
        vocab[token] = vocab.get(token, 0) + 1
    return vocab

vocab = get_vocab(corpus)

for word, freq in vocab.items():
    print(f"{freq}x  {word}")

3x  l o w </w>
1x  l o w e r </w>
1x  l o w e s t </w>
1x  n e w e r </w>
1x  w i d e r </w>


### **Step 3 — Count All Adjacent Pairs**

In [3]:
def get_pairs(vocab):
    pairs = {}
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair = (symbols[i], symbols[i+1])
            pairs[pair] = pairs.get(pair, 0) + freq
    return pairs

pairs = get_pairs(vocab)

# Sort and print
for pair, freq in sorted(pairs.items(), key=lambda x: -x[1]):
    print(f"{freq}x  {pair}")

5x  ('l', 'o')
5x  ('o', 'w')
3x  ('w', '</w>')
3x  ('w', 'e')
3x  ('e', 'r')
3x  ('r', '</w>')
1x  ('e', 's')
1x  ('s', 't')
1x  ('t', '</w>')
1x  ('n', 'e')
1x  ('e', 'w')
1x  ('w', 'i')
1x  ('i', 'd')
1x  ('d', 'e')


### **Step 4 — Merge the Best Pair**

In [4]:
import re

def merge_vocab(pair, vocab):
    new_vocab = {}
    # Create pattern to find the pair (with space between)
    bigram = " ".join(pair)
    replacement = "".join(pair)  # merged token (no space)

    for word in vocab:
        # Replace the pair with merged version
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]

    return new_vocab

# Merge the most frequent pair: ('l', 'o') → 'lo'
best_pair = ('l', 'o')
vocab = merge_vocab(best_pair, vocab)

for word, freq in vocab.items():
    print(f"{freq}x  {word}")

3x  lo w </w>
1x  lo w e r </w>
1x  lo w e s t </w>
1x  n e w e r </w>
1x  w i d e r </w>


### **Step 5 — Run Multiple Merge Rounds**

In [5]:
def train_bpe(corpus, num_merges):
    vocab = get_vocab(corpus)
    merge_rules = []

    print("Initial vocab:")
    for word, freq in vocab.items():
        print(f"  {freq}x  {word}")
    print()

    for i in range(num_merges):
        pairs = get_pairs(vocab)
        if not pairs:
            break

        # Pick the most frequent pair
        best = max(pairs, key=pairs.get)
        merge_rules.append(best)

        # Merge it
        vocab = merge_vocab(best, vocab)

        print(f"Merge {i+1}: {best[0]} + {best[1]} → {''.join(best)}")
        for word, freq in vocab.items():
            print(f"  {freq}x  {word}")
        print()

    return vocab, merge_rules


vocab, rules = train_bpe(corpus, num_merges=5)

Initial vocab:
  3x  l o w </w>
  1x  l o w e r </w>
  1x  l o w e s t </w>
  1x  n e w e r </w>
  1x  w i d e r </w>

Merge 1: l + o → lo
  3x  lo w </w>
  1x  lo w e r </w>
  1x  lo w e s t </w>
  1x  n e w e r </w>
  1x  w i d e r </w>

Merge 2: lo + w → low
  3x  low </w>
  1x  low e r </w>
  1x  low e s t </w>
  1x  n e w e r </w>
  1x  w i d e r </w>

Merge 3: low + </w> → low</w>
  3x  low</w>
  1x  low e r </w>
  1x  low e s t </w>
  1x  n e w e r </w>
  1x  w i d e r </w>

Merge 4: e + r → er
  3x  low</w>
  1x  low er </w>
  1x  low e s t </w>
  1x  n e w er </w>
  1x  w i d er </w>

Merge 5: er + </w> → er</w>
  3x  low</w>
  1x  low er</w>
  1x  low e s t </w>
  1x  n e w er</w>
  1x  w i d er</w>



### **Step 6 — Tokenize a New Word**

In [6]:
def tokenize(word, merge_rules):
    # Start with characters + end marker
    tokens = list(word) + ["</w>"]
    tokens = " ".join(tokens)

    print(f"Start:  {tokens}")

    # Apply each merge rule in order
    for rule in merge_rules:
        bigram = " ".join(rule)
        merged = "".join(rule)
        tokens = tokens.replace(bigram, merged)
        print(f"Apply ({rule[0]} + {rule[1]}):  {tokens}")

    return tokens.split()


print("\nTokenizing 'lower':")
result = tokenize("lower", rules)
print(f"\nFinal tokens: {result}")

print("\nTokenizing 'lowest':")
result = tokenize("lowest", rules)
print(f"\nFinal tokens: {result}")


Tokenizing 'lower':
Start:  l o w e r </w>
Apply (l + o):  lo w e r </w>
Apply (lo + w):  low e r </w>
Apply (low + </w>):  low e r </w>
Apply (e + r):  low er </w>
Apply (er + </w>):  low er</w>

Final tokens: ['low', 'er</w>']

Tokenizing 'lowest':
Start:  l o w e s t </w>
Apply (l + o):  lo w e s t </w>
Apply (lo + w):  low e s t </w>
Apply (low + </w>):  low e s t </w>
Apply (e + r):  low e s t </w>
Apply (er + </w>):  low e s t </w>

Final tokens: ['low', 'e', 's', 't', '</w>']
